# Phase 1: Document Vectorization

This notebook demonstrates the first stage of the RAG pipeline: **Ingestion and Storage**.

We will cover:
1. **Extraction**: Reading text from a PDF.
2. **Chunking**: Splitting text into manageable segments.
3. **Embedding**: Converting text into semantic vectors.
4. **Indexing**: Saving segments to a local ChromaDB store.

In [ ]:
import os
import sys

# Add the src directory to the python path so we can import our modules
sys.path.append(os.path.abspath('../src'))

from ingest.parser import extract_text_from_pdf
from ingest.chunker import chunk_text
from rag.retriever import VectorRetriever

print("Modules loaded successfully.")

## 1. Text Extraction

We'll use `pdfplumber` (wrapped in our `parser` module) to extract raw text from a sample PDF.

In [ ]:
pdf_path = "../data/raw/sample.pdf"

if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found. Please ensure a sample PDF exists in data/raw/")
else:
    text = extract_text_from_pdf(pdf_path)
    print(f"Extracted {len(text)} characters from the PDF.")
    print("\n--- Sample Text (First 500 chars) ---")
    print(text[:500])

## 2. Text Chunking

Raw text is too long for LLMs to process efficiently. We split it into smaller overlapping chunks.

In [ ]:
chunks = chunk_text(text, chunk_size=500, chunk_overlap=100)
print(f"Split text into {len(chunks)} chunks.")

print("\n--- Chunk 1 ---")
print(chunks[0])
if len(chunks) > 1:
    print("\n--- Chunk 2 ---")
    print(chunks[1])

## 3. Vector Storage (ChromaDB)

Now we convert these chunks into embeddings using a local `sentence-transformer` model and save them to our local ChromaDB.

In [ ]:
retriever = VectorRetriever()

# Add chunks to store
metadatas = [{"source": "sample.pdf"} for _ in chunks]
retriever.add_chunks(chunks, metadatas=metadatas)

print(f"Successfully indexed {retriever.get_collection_count()} items in the vector store.")

## 4. Verification

Let's check if we can retrieve something back (even without full search logic implemented yet).

In [ ]:
results = retriever.collection.peek(limit=1)
print("Peek at one stored document:")
print(results['documents'][0])